# Driven chirality-flip toy model

**Package role:** Manual companion notebook

**Source of truth:** `manual/sections/04e_recurrent_systems.tex`

This notebook gives a toy recurrent collision model for chirality flips, push/pull response, and mean chirality polarization under unbiased and driven reclosure schedules.


In [ ]:

import math
import random
from dataclasses import dataclass
from typing import List, Tuple

@dataclass
class Duon:
    x: float
    v: float
    chirality: int
    phase: float
    burden: float

def wrap_phase(p: float) -> float:
    return p % (2 * math.pi)

def make_population(n: int, length: float, seed: int = 11) -> List[Duon]:
    random.seed(seed)
    return [
        Duon(
            x=random.uniform(0, length),
            v=random.choice([-1.0, 1.0]) * random.uniform(0.15, 0.95),
            chirality=random.choice([-1, 1]),
            phase=random.uniform(0, 2 * math.pi),
            burden=random.uniform(0.15, 1.1),
        )
        for _ in range(n)
    ]

def resonance_score(a: Duon, b: Duon, field_bias: float) -> float:
    phase_term = 0.5 * (1 + math.cos((a.phase - b.phase) - math.pi))
    burden_term = math.exp(-abs(a.burden - b.burden) / 0.35)
    speed_term = min(abs(a.v - b.v) / 1.5, 1.0)
    opposite_term = 1.0 if a.chirality != b.chirality else 0.8
    field_align = 0.5 * (1 + field_bias * (a.chirality - b.chirality) / 2)
    return max(
        0.0,
        min(
            1.0,
            phase_term
            * burden_term
            * (0.55 + 0.45 * speed_term)
            * opposite_term
            * (0.75 + 0.5 * field_align),
        ),
    )

def collide(a: Duon, b: Duon, field_bias: float) -> Tuple[int, int]:
    score = resonance_score(a, b, field_bias)

    # Transport-layer scattering
    va, vb = a.v, b.v
    a.v, b.v = 0.9 * vb, 0.9 * va
    a.phase = wrap_phase(a.phase + 0.55 * (b.chirality + 0.6 * field_bias))
    b.phase = wrap_phase(b.phase + 0.55 * (a.chirality - 0.6 * field_bias))
    a.burden = min(max(a.burden + 0.08 * random.uniform(-1, 1), 0.0), 1.5)
    b.burden = min(max(b.burden + 0.08 * random.uniform(-1, 1), 0.0), 1.5)

    flips = 0
    breakups = 0

    # Resonant chirality flip / biasing
    if score > 0.34:
        pflip = min(0.85, 0.18 + 1.15 * (score - 0.34))
        if random.random() < pflip:
            if random.random() < (0.5 + 0.28 * field_bias):
                a.chirality = 1
                b.chirality = 1 if random.random() < 0.75 else -1
            else:
                a.chirality = -1
                b.chirality = -1 if random.random() < 0.75 else 1
            flips = 1

    # Stronger resonance can reclose / restructure
    if score > 0.72 and random.random() < 0.22 + 0.5 * (score - 0.72):
        a.phase = wrap_phase(a.phase + math.pi / 2)
        b.phase = wrap_phase(b.phase - math.pi / 2)
        a.burden = 0.5 * a.burden + 0.2
        b.burden = 0.5 * b.burden + 0.2
        breakups = 1

    return flips, breakups

def simulate(
    n=100,
    steps=900,
    dt=0.22,
    length=70.0,
    interaction_radius=0.85,
    field_bias_schedule=None,
    seed: int = 11,
):
    if field_bias_schedule is None:
        field_bias_schedule = lambda t: 0.0

    duons = make_population(n, length, seed=seed)

    times = []
    mean_chirality = []
    mean_pushpull = []
    flips = []
    breakups = []
    collisions = []
    field_values = []

    for step in range(steps):
        t = step * dt
        field_bias = max(min(field_bias_schedule(t), 1.0), -1.0)

        for d in duons:
            d.x += dt * (d.v + 0.24 * field_bias * d.chirality)
            d.x %= length
            d.phase = wrap_phase(d.phase + dt * (0.75 + 0.32 * d.burden))

        step_flips = 0
        step_breakups = 0
        step_collisions = 0

        for i in range(len(duons)):
            for j in range(i + 1, len(duons)):
                a, b = duons[i], duons[j]
                dx = abs(a.x - b.x)
                dx = min(dx, length - dx)
                if dx < interaction_radius:
                    step_collisions += 1
                    f, br = collide(a, b, field_bias)
                    step_flips += f
                    step_breakups += br

        chir = sum(d.chirality for d in duons) / len(duons)
        pp = sum(d.chirality * math.copysign(1, d.v) for d in duons) / len(duons)

        times.append(t)
        mean_chirality.append(chir)
        mean_pushpull.append(pp)
        flips.append(step_flips)
        breakups.append(step_breakups)
        collisions.append(step_collisions)
        field_values.append(field_bias)

    return {
        "times": times,
        "mean_chirality": mean_chirality,
        "mean_pushpull": mean_pushpull,
        "flips": flips,
        "breakups": breakups,
        "collisions": collisions,
        "field": field_values,
    }

def summarize(run):
    return {
        "avg_chirality": round(sum(run["mean_chirality"]) / len(run["mean_chirality"]), 4),
        "avg_pushpull": round(sum(run["mean_pushpull"]) / len(run["mean_pushpull"]), 4),
        "total_flips": int(sum(run["flips"])),
        "total_breakups": int(sum(run["breakups"])),
        "total_collisions": int(sum(run["collisions"])),
        "peak_abs_chirality": round(max(abs(x) for x in run["mean_chirality"]), 4),
    }

if __name__ == "__main__":
    baseline = simulate(field_bias_schedule=lambda t: 0.0)
    driven = simulate(field_bias_schedule=lambda t: 0.95 * math.sin(2 * math.pi * t / 28.0))
    print("baseline", summarize(baseline))
    print("driven  ", summarize(driven))


In [ ]:
baseline = simulate(field_bias_schedule=lambda t: 0.0)
driven = simulate(field_bias_schedule=lambda t: 0.95 * math.sin(2 * math.pi * t / 28.0))

baseline_summary = summarize(baseline)
driven_summary = summarize(driven)

baseline_summary['flips_per_collision'] = round(baseline_summary['total_flips'] / baseline_summary['total_collisions'], 5)
driven_summary['flips_per_collision'] = round(driven_summary['total_flips'] / driven_summary['total_collisions'], 5)

baseline_summary, driven_summary

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(baseline['times'], baseline['mean_chirality'], label='No external bias')
ax.plot(driven['times'], driven['mean_chirality'], label='Driven bias')
ax.set_title('Duon chirality polarization')
ax.set_xlabel('Time')
ax.set_ylabel('Mean chirality')
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(baseline['times'], baseline['mean_pushpull'], label='No external bias')
ax.plot(driven['times'], driven['mean_pushpull'], label='Driven bias')
ax.set_title('Effective push/pull response')
ax.set_xlabel('Time')
ax.set_ylabel('Push/pull proxy')
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(baseline['times'], baseline['flips'], label='No external bias')
ax.plot(driven['times'], driven['flips'], label='Driven bias')
ax.set_title('Resonant chirality flips')
ax.set_xlabel('Time')
ax.set_ylabel('Flip events per step')
ax.legend()
plt.show()

The driven schedule produces more flips, a higher flip probability per collision, and larger chirality excursions than the unbiased run. A fuller AOD-style upgrade would derive the flip condition from explicit backlog/reconciliation order rather than from the current toy resonance rule.